# 01 · Predicción de precio (CABA, venta, USD)

Primer modelo sobre los datos del scraper. Usa las mismas funciones que la sección **Modelos** de la interfaz (`ml/zpml.py`).

- **Datos:** `data/listings.jsonl` (última versión de cada aviso) + `data/details.jsonl` si existe.
- **Filtro:** CABA (ubicación terminada en *Capital Federal* o en un barrio porteño), venta, precio publicado en **USD**. Los avisos en pesos se descartan (opción: convertir con el dólar MEP).
- **Objetivo:** `log(precio)`; las métricas se reportan en USD.
- **Modelos:** Ridge (baseline) y LightGBM; SHAP para explicar.

Kernel: el Python de `ml/.venv` (ver README, sección *Modelos*).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'ml'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import zpml

# Parámetros
OPERACION = 'venta'
MONEDA = 'USD'          # 'USD' | 'ARS' | 'convertir_a_USD'
DOLAR_MEP = 1450        # sólo si MONEDA == 'convertir_a_USD'
TIPO = 'departamento'   # 'todos', 'departamento', 'casa', 'ph', ...
TEST_SIZE = 0.2
SEED = 42

## Carga y filtro

In [ ]:
df = zpml.load()                      # sólo CABA
print(f'{len(df):,} avisos de CABA')
df['operacion'].value_counts(), df['price_currency'].value_counts(dropna=False)

In [ ]:
d = zpml.price_frame(df, OPERACION, MONEDA, DOLAR_MEP, TIPO)
print(f'{len(d):,} avisos con precio y m² válidos')
d[['barrio','tipo_prop','price_x','sup','ppm2']].describe(include='all').T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
np.log10(d['price_x']).hist(bins=60, ax=ax[0]); ax[0].set_title('log10 precio')
d.groupby('barrio')['ppm2'].median().sort_values().tail(25).plot.barh(ax=ax[1], title='USD/m² mediano (top 25 barrios)')
plt.tight_layout()

## Variables

In [ ]:
X, nums, cats = zpml.build_features(d)
y = np.log(d['price_x'].values)
print('numéricas:', nums); print('categóricas:', cats)
X.isna().mean().sort_values(ascending=False).head(15)

In [ ]:
from sklearn.model_selection import train_test_split
tr, te = train_test_split(np.arange(len(d)), test_size=TEST_SIZE, random_state=SEED)
Xtr, Xte = X.iloc[tr], X.iloc[te]
price_te = d['price_x'].values[te]

def metricas(real, pred):
    ape = np.abs(pred - real) / real
    from sklearn.metrics import r2_score
    return {'MAE_USD': np.mean(np.abs(pred - real)), 'MAPE_mediana_%': np.median(ape) * 100,
            'dentro_10%_%': np.mean(ape < .1) * 100, 'R2_log': r2_score(np.log(real), np.log(pred))}

## Baseline: Ridge

In [ ]:
import run as mlrun   # ml/run.py
obj = lambda Z: Z.astype({c: object for c in cats})
ridge = mlrun.ridge_pipe(nums, cats, alpha=1.0).fit(obj(Xtr), y[tr])
res = {'Ridge': metricas(price_te, np.exp(ridge.predict(obj(Xte))))}
pd.DataFrame(res).T

## LightGBM

In [ ]:
try:
    import lightgbm as lgb
    model = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.8, min_child_samples=20, verbose=-1, random_state=SEED)
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor as HGB
    model = HGB(max_iter=800, learning_rate=0.05, categorical_features='from_dtype', random_state=SEED)
model.fit(Xtr, y[tr])
pred = np.exp(model.predict(Xte))
res[type(model).__name__] = metricas(price_te, pred)
pd.DataFrame(res).T

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(price_te, pred, s=6, alpha=.4); lim = np.percentile(price_te, [1, 99]); ax.plot(lim, lim, 'r--')
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('real'); ax.set_ylabel('predicción')

## Explicación (SHAP)

In [ ]:
try:
    import shap
    sample = Xte.sample(min(1000, len(Xte)), random_state=0)
    shap.summary_plot(shap.TreeExplainer(model).shap_values(sample), sample, max_display=20)
except Exception as e:
    print('SHAP no disponible:', e)

## Residuos por barrio
Positivo = el mercado publica más caro de lo que predice el modelo.

In [ ]:
out = d.iloc[te][['id','url','barrio','price_x','sup','ambientes']].assign(pred=pred)
out['residuo_%'] = (out['price_x'] / out['pred'] - 1) * 100
display(out.groupby('barrio')['residuo_%'].agg(['median','size']).sort_values('median'))
out.sort_values('residuo_%').head(20)   # los más baratos vs. el modelo